
# Zadanie: ulepsz klasyfikator oparty na sieci neuronowej

## Cel projektu
Otrzymujesz **działającą, ale celowo nieoptymalną sieć neuronową** do klasyfikacji binarnej.  
Twoim zadaniem jest przeprowadzić serię eksperymentów i **poprawić jakość modelu**, jednocześnie kontrolując przeuczenie.

Notebook zawiera kompletny pipeline:
- wczytanie danych,
- analizę podstawową,
- podział na train / validation / test,
- standaryzację cech,
- budowę sieci neuronowej,
- trening z walidacją,
- predykcję,
- metryki klasyfikacyjne,
- macierz pomyłek,
- krzywą ROC,
- wykresy przebiegu treningu.

## Twoje zadanie
Spróbuj poprawić model, eksperymentując m.in. z:

1. **Liczbą warstw ukrytych** i liczbą neuronów.
2. Funkcjami aktywacji: `relu`, `tanh`, `elu`, `selu`.
3. **Learning rate**.
4. Optymalizatorem: `Adam`, `RMSprop`, `SGD`, `AdamW`.
5. Liczbą epok.
6. Wielkością `batch_size`.
7. `Dropout`.
8. Regularyzacją `L1` / `L2`.
9. `BatchNormalization`.
10. `EarlyStopping`.
11. `ReduceLROnPlateau`.
12. Inną inicjalizacją wag.
13. Doborem progu decyzyjnego zamiast sztywnego `0.5`.
14. Walidacją krzyżową lub wielokrotnym powtórzeniem eksperymentu.

## Warunek
Nie oceniaj modelu wyłącznie przez `accuracy`. Porównuj co najmniej:

- accuracy,
- precision,
- recall,
- F1-score,
- ROC-AUC,
- macierz pomyłek.

## Pytania do odpowiedzi
Na końcu eksperymentu odpowiedz:

- Co zmieniłeś?
- Dlaczego?
- Która zmiana dała największą poprawę?
- Czy poprawa na zbiorze walidacyjnym przeniosła się na zbiór testowy?
- Czy pojawiło się przeuczenie?
- Jaki model wybierasz jako końcowy i dlaczego?

### Ważne
Bazowy model **ma działać, ale nie jest wzorcem dobrej optymalizacji**.  
Nie kopiuj parametrów bez eksperymentów. Traktuj go jako punkt startowy.

> Dataset pochodzi ze zbioru Breast Cancer Wisconsin dostępnego w scikit-learn.  
> Projekt ma charakter wyłącznie edukacyjny i nie służy do podejmowania decyzji medycznych.



## 1. Import bibliotek i ustawienie losowości


In [ ]:
import os
import sys
import random
import subprocess
import importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve
)

# Google Colab zwykle ma TensorFlow już zainstalowany.
# W lokalnym Jupyterze notebook spróbuje doinstalować go automatycznie.
if importlib.util.find_spec("tensorflow") is None:
    print("TensorFlow nie jest zainstalowany. Instaluję pakiet...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)


## 2. Wczytanie danych
Używamy zbioru **Breast Cancer Wisconsin**. Jest to klasyfikacja binarna.


In [ ]:

data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("Liczba obserwacji:", len(X))
print("Liczba cech:", X.shape[1])
print("\nKlasy:")
for i, name in enumerate(data.target_names):
    print(f"  {i}: {name}")

display(X.head())



## 3. Krótka analiza danych


In [ ]:

print("Braki danych:", X.isna().sum().sum())
print("\nRozkład klas:")
print(y.value_counts().sort_index())

print("\nUdział procentowy klas:")
display((y.value_counts(normalize=True).sort_index() * 100).round(2).rename("procent"))

display(X.describe().T.head(10))


In [ ]:

counts = y.value_counts().sort_index()

plt.figure(figsize=(6, 4))
plt.bar([data.target_names[i] for i in counts.index], counts.values)
plt.title("Rozkład klas")
plt.xlabel("Klasa")
plt.ylabel("Liczba obserwacji")
plt.show()



## 4. Podział na train / validation / test

Zastosujemy:
- 70% trening,
- 15% walidacja,
- 15% test.

Używamy `stratify`, aby zachować proporcje klas.


In [ ]:

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=SEED,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)



## 5. Standaryzacja

`StandardScaler` uczymy **wyłącznie na zbiorze treningowym**, aby uniknąć wycieku informacji.


In [ ]:

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Średnia pierwszej cechy po skalowaniu:", X_train_scaled[:, 0].mean().round(4))
print("Odchylenie pierwszej cechy:", X_train_scaled[:, 0].std().round(4))



# 6. Model bazowy

Poniższa architektura jest **celowo tylko punktem startowym**.

Nie ma tutaj:
- Dropout,
- BatchNormalization,
- regularyzacji,
- EarlyStopping,
- harmonogramu learning rate,
- strojenia progu klasyfikacji.

Dodatkowo learning rate, batch size i liczba epok są ustawione arbitralnie.


In [ ]:

def build_baseline_model(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation="relu"),
        layers.Dense(8, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    optimizer = keras.optimizers.Adam(learning_rate=0.01)

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="auc")
        ]
    )
    return model

model = build_baseline_model(X_train_scaled.shape[1])
model.summary()



## 7. Trening bazowej sieci
Celowo **bez callbacków optymalizacyjnych**.


In [ ]:

history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=25,
    batch_size=128,
    verbose=1
)



## 8. Wykresy przebiegu uczenia
Szukaj sygnałów underfittingu i overfittingu.


In [ ]:

history_df = pd.DataFrame(history.history)
history_df["epoch"] = np.arange(1, len(history_df) + 1)

plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["loss"], label="train loss")
plt.plot(history_df["epoch"], history_df["val_loss"], label="validation loss")
plt.title("Funkcja kosztu")
plt.xlabel("Epoka")
plt.ylabel("Binary cross-entropy")
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["accuracy"], label="train accuracy")
plt.plot(history_df["epoch"], history_df["val_accuracy"], label="validation accuracy")
plt.title("Accuracy")
plt.xlabel("Epoka")
plt.ylabel("Accuracy")
plt.legend()
plt.show()



## 9. Ocena na zbiorze walidacyjnym


In [ ]:

val_results = model.evaluate(X_val_scaled, y_val, verbose=0)

print("Wyniki walidacyjne:")
for name, value in zip(model.metrics_names, val_results):
    print(f"{name:>12}: {value:.4f}")



## 10. Predykcja na zbiorze testowym
Domyślnie stosujemy próg `0.5`. To również może być przedmiotem optymalizacji.


In [ ]:

y_prob = model.predict(X_test_scaled, verbose=0).ravel()

THRESHOLD = 0.5
y_pred = (y_prob >= THRESHOLD).astype(int)

predictions = pd.DataFrame({
    "y_true": y_test.to_numpy(),
    "probability_class_1": y_prob,
    "y_pred": y_pred
})

display(predictions.head(15))



## 11. Metryki końcowe


In [ ]:

metrics_baseline = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_prob)
}

display(pd.DataFrame([metrics_baseline]).round(4))

print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=data.target_names,
    digits=4
))



## 12. Macierz pomyłek


In [ ]:

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=data.target_names
)

disp.plot(values_format="d")
plt.title("Macierz pomyłek — model bazowy")
plt.show()



## 13. Krzywa ROC


In [ ]:

fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc_value = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"Sieć neuronowa, AUC = {auc_value:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Losowy klasyfikator")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Krzywa ROC — model bazowy")
plt.legend()
plt.show()



## 14. Rozkład prawdopodobieństw predykcji
To pomaga zobaczyć, czy próg `0.5` rzeczywiście ma sens.


In [ ]:

plt.figure(figsize=(8, 5))
plt.hist(y_prob[y_test.to_numpy() == 0], bins=15, alpha=0.7, label=data.target_names[0])
plt.hist(y_prob[y_test.to_numpy() == 1], bins=15, alpha=0.7, label=data.target_names[1])
plt.axvline(THRESHOLD, linestyle="--", label=f"próg = {THRESHOLD}")
plt.xlabel("Prawdopodobieństwo klasy 1")
plt.ylabel("Liczba przypadków")
plt.title("Rozkład prawdopodobieństw predykcji")
plt.legend()
plt.show()



# 15. Strefa eksperymentów kursanta

Zbuduj **nową wersję modelu**, nie nadpisując modelu bazowego.

### Minimalny plan eksperymentu
Przetestuj przynajmniej 5 konfiguracji.

Przykładowa tabela:

| Eksperyment | Warstwy | LR | Batch | Epochs | Dropout | Val AUC | Test AUC | Test F1 |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| baseline | 32-8 | 0.01 | 128 | 25 | 0.0 | ... | ... | ... |
| exp_1 | ... | ... | ... | ... | ... | ... | ... | ... |

### Spróbuj odpowiedzieć:
- Czy większa sieć zawsze daje lepszy wynik?
- Czy mniejszy learning rate stabilizuje trening?
- Czy więcej epok poprawia test, czy tylko train?
- Czy `Dropout` pomaga?
- Czy `EarlyStopping` zatrzymuje trening w sensownym momencie?
- Czy próg `0.5` jest najlepszy dla F1 lub recall?


In [ ]:

# ============================================
# MIEJSCE NA MODEL KURSANTA
# ============================================

def build_student_model(input_dim):
    # TODO: zmień architekturę i hiperparametry.
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),

        # PRZYKŁAD:
        # layers.Dense(64, activation="relu"),
        # layers.Dropout(0.2),
        # layers.Dense(32, activation="relu"),

        layers.Dense(32, activation="relu"),
        layers.Dense(8, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    optimizer = keras.optimizers.Adam(
        learning_rate=0.01  # TODO: zmień
    )

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="auc")
        ]
    )

    return model


student_model = build_student_model(X_train_scaled.shape[1])
student_model.summary()



## 16. Trening modelu kursanta
Dodaj callbacki, jeśli chcesz. Celowo pozostawiono tu prosty szkielet.


In [ ]:

# PRZYKŁADOWE CALLBACKI DO WŁĄCZENIA:
#
# callbacks = [
#     keras.callbacks.EarlyStopping(
#         monitor="val_loss",
#         patience=10,
#         restore_best_weights=True
#     ),
#     keras.callbacks.ReduceLROnPlateau(
#         monitor="val_loss",
#         factor=0.5,
#         patience=4
#     )
# ]
#
# Następnie przekaż: callbacks=callbacks

student_history = student_model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=25,       # TODO
    batch_size=128,  # TODO
    verbose=1
)



## 17. Porównanie modelu kursanta z baseline


In [ ]:

student_prob = student_model.predict(X_test_scaled, verbose=0).ravel()
student_pred = (student_prob >= 0.5).astype(int)

metrics_student = {
    "accuracy": accuracy_score(y_test, student_pred),
    "precision": precision_score(y_test, student_pred),
    "recall": recall_score(y_test, student_pred),
    "f1": f1_score(y_test, student_pred),
    "roc_auc": roc_auc_score(y_test, student_prob)
}

comparison = pd.DataFrame(
    [metrics_baseline, metrics_student],
    index=["baseline", "student_model"]
)

display(comparison.round(4))



# 18. Podsumowanie do oddania

Uzupełnij:

### 1. Model bazowy
- Test accuracy:
- Test precision:
- Test recall:
- Test F1:
- Test ROC-AUC:

### 2. Najlepszy model
- Architektura:
- Optimizer:
- Learning rate:
- Batch size:
- Liczba epok:
- Dropout / regularyzacja:
- Callbacki:
- Próg klasyfikacji:

### 3. Wyniki najlepszego modelu
- Test accuracy:
- Test precision:
- Test recall:
- Test F1:
- Test ROC-AUC:

### 4. Wnioski
Napisz 5–10 zdań o tym:
- które hiperparametry miały największy wpływ,
- gdzie pojawił się underfitting / overfitting,
- czy poprawa walidacji przełożyła się na test,
- jaki kompromis między precision i recall wybrałeś.
